# 0.5B AIT Tape Compiler LoRA - Mixed Auxiliary Tasks

This Colab notebook fine-tunes a small instruct model to compile natural-language instructions into 4-character AIT v0 packets, with auxiliary target/action extraction rows mixed into training.

Target task:

```text
natural instruction -> AIT code
scan context 4 for xss urgently -> s4x9
```

The goal is stable protocol emission, not deep reasoning.

In [ ]:
# Runtime config
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
TRAIN_ROWS = 2400
EVAL_ROWS = 200
EPOCHS = 3
DATA_MODE = "mixed"
MAX_LEN = 128
REPO_URL = "https://github.com/kagioneko/ai-instruction-tape.git"
REPO_DIR = "/content/ai-instruction-tape"

## 1. Install Dependencies

In [ ]:
!nvidia-smi
# Colab may preinstall an old torchao. PEFT checks it if present, but this experiment does not use torchao.
!pip uninstall -y -q torchao
!pip install -q -U transformers datasets accelerate peft
!rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -e .

## 2. Generate Synthetic Data

In [ ]:
!PYTHONPATH=src python experiments/0_5b_tape_compiler/generate_dataset.py --out experiments/0_5b_tape_compiler/data/train.jsonl --count {TRAIN_ROWS} --seed 42 --mode {DATA_MODE}
!PYTHONPATH=src python experiments/0_5b_tape_compiler/generate_dataset.py --out experiments/0_5b_tape_compiler/data/eval.jsonl --count {EVAL_ROWS} --seed 7 --mode ait
!head -n 3 experiments/0_5b_tape_compiler/data/train.jsonl

## 3. Load Model and Dataset

In [ ]:
import json
from pathlib import Path
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorForLanguageModeling, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

SYSTEM = "You are an AIT v0 compiler. Output only one 4-character lowercase ASCII code."

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

train_path = Path("experiments/0_5b_tape_compiler/data/train.jsonl")
eval_path = Path("experiments/0_5b_tape_compiler/data/eval.jsonl")
if not train_path.exists() or not eval_path.exists():
    print("Dataset files not found; generating them now.")
    !PYTHONPATH=src python experiments/0_5b_tape_compiler/generate_dataset.py --out experiments/0_5b_tape_compiler/data/train.jsonl --count {TRAIN_ROWS} --seed 42 --mode {DATA_MODE}
    !PYTHONPATH=src python experiments/0_5b_tape_compiler/generate_dataset.py --out experiments/0_5b_tape_compiler/data/eval.jsonl --count {EVAL_ROWS} --seed 7 --mode ait

ds = load_dataset("json", data_files={
    "train": "experiments/0_5b_tape_compiler/data/train.jsonl",
    "eval": "experiments/0_5b_tape_compiler/data/eval.jsonl",
})

## 4. Format Chat Prompts

In [ ]:
def to_text(row, include_answer=True):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": row["input"]},
    ]
    if include_answer:
        messages.append({"role": "assistant", "content": row["output"]})
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=not include_answer)

def tokenize(row):
    text = to_text(row, include_answer=True)
    return tokenizer(text, truncation=True, max_length=MAX_LEN)

tok = ds.map(tokenize, remove_columns=ds["train"].column_names)
tok

## 5. Train LoRA Adapter

In [ ]:
import inspect

training_kwargs = dict(
    output_dir="experiments/0_5b_tape_compiler/ait-0_5b-lora-mixed",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=EPOCHS,
    learning_rate=2e-4,
    logging_steps=20,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none",
)
strategy_arg = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments).parameters else "evaluation_strategy"
training_kwargs[strategy_arg] = "epoch"
args = TrainingArguments(**training_kwargs)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok["train"],
    eval_dataset=tok["eval"],
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
trainer.train()
trainer.save_model("experiments/0_5b_tape_compiler/ait-0_5b-lora-mixed/final")

## 6. Generate Predictions

In [ ]:
import re

def clean_code(text):
    text = text.strip().lower()
    m = re.search(r"[a-z0-9]{4}", text)
    return m.group(0) if m else text.split()[0] if text.split() else ""

model.eval()
preds = []
for row in ds["eval"]:
    prompt = to_text(row, include_answer=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    preds.append(clean_code(generated))

pred_path = "experiments/0_5b_tape_compiler/predictions_colab.txt"
with open(pred_path, "w", encoding="utf-8") as f:
    for p in preds:
        f.write(p + "\n")

preview = ds["eval"].select(range(min(10, len(ds["eval"]))))
list(zip(preview["input"], preview["output"], preds[:10]))

## 7. Score Outputs

In [ ]:
!PYTHONPATH=src python experiments/0_5b_tape_compiler/evaluate_outputs.py --gold experiments/0_5b_tape_compiler/data/eval.jsonl --pred experiments/0_5b_tape_compiler/predictions_colab.txt

## Next Tweaks

- If `valid` is low, increase epochs or make the system prompt stricter.
- If only `target` is low, add more context-number phrasing to `generate_dataset.py`.
- If `exact` is high on synthetic data, add a hand-written eval set before claiming anything.

## 8. Download Result Files

In [ ]:
from google.colab import files

files.download("/content/ai-instruction-tape/experiments/0_5b_tape_compiler/predictions_colab.txt")
files.download("/content/ai-instruction-tape/experiments/0_5b_tape_compiler/data/eval.jsonl")